# Test de la fonction d'association de station à des échangeurs

In [1]:
from datetime import date
from pathlib import Path

from enum import StrEnum
import sys
import folium
import json
import geopandas as gpd
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))

from constants import TILES_IGN, ATTR_IGN

CHUNK_SIZE: int = 20 #0
SAMPLES: int = 288  # 5 min
SATURE_H: int = 45  # minimum duration (min) of saturation to have a saturated hour
MAX_SESSION_DURATION_HOURS = 24

ID_POC: str = "id_pdc_itinerance"
ID_STATION: str = "id_station_itinerance"
ID_POOL: str = "id_pool"
SATURATION_RATIO = 0.1
OVERLOAD_RATIO = 0.2
MIN_POWER = 75
RAYON_ECHANGEUR = 3000
BUFFER_ECHANGEUR = 100

#day = date(2026,7, 5)
#day = date(2026,7, 14)
#day = date(2026,7, 25)
day = date(2026,8, 1)
#day = date(2026,8, 18)
#day = date(2026,8, 25)
date_file = f"{day.year}{day.month:02d}{day.day:02d}"

data_quali = "../data/"

class NaturePool(StrEnum):
    AIRE = "A"
    ECHANGEUR = "E"

In [2]:
def stations_echangeurs(
    stations: gpd.GeoDataFrame,
    echangeurs: gpd.GeoDataFrame,
    rayon_echangeur: float
) -> gpd.GeoDataFrame:
    """Return the nearest echangeur for each station within a maximum distance."""
    stations = stations.drop_duplicates(subset=[ID_STATION])
    ech_geo = gpd.GeoDataFrame(echangeurs, geometry=gpd.points_from_xy(echangeurs["longitude"], echangeurs["latitude"]), crs="EPSG:4326")
    stations_geo = gpd.GeoDataFrame(
        stations, 
        geometry=gpd.points_from_xy(stations["longitude"], stations["latitude"]), 
        crs="EPSG:4326")
   
    ech_station = gpd.sjoin_nearest(
        stations_geo[[ID_STATION, "geometry"]].to_crs(epsg=2154),
        ech_geo[[ID_POOL, "geometry"]].to_crs(epsg=2154),
        how="inner",
        max_distance=rayon_echangeur,
        distance_col="distance",
    )
    return ech_station[[ID_STATION, "geometry", ID_POOL, "distance"]].reset_index(drop=True).to_crs(epsg=4326)

In [3]:
# test stations_echangeurs

#example_json = ech_station[ech_station[ID_POOL].isin(["E000001", "E000003"])].sort_values(by=ID_POOL).reset_index(drop=True)[[ID_STATION, ID_POOL]].to_json(orient="split", index=None)
#example = json.loads(example_json)
stations = pd.DataFrame(
    [['FRHPCPNF059704TOTEMSA', 2.522993, 49.000044],
    ['FRGYMP735571', 2.553983, 49.089106],
    ['FRFASP11568718', 2.551434, 49.073179],
    ['FRTSLP436970', 2.515686, 48.991119],
    ['FRELCP12953864', 2.515492, 48.98999],
    ['FRIOYP13530862', 2.548005, 49.089443],
    ['FRPD1PKFCWTZ', 2.556275, 49.088774],
    ['FRFASP11568719', 2.552983, 49.071747],
    ['FRECHPCRXMB', 2.555722, 49.090025],
    ['FRVIAP152005', 2.528647, 49.093699],
    ['FRELCP13050707', 2.531782, 49.096637],
    ['FRHPCPNF059705TOTEM', 2.537335, 49.00931],
    ['FRGYMP735569', 2.560844717, 49.008585705],
    ['FRELCP12953963', 2.517808, 48.998928],
    ['FRMILP11437562', 2.549234, 49.088313],
    ['FRVIAP122064', 2.514075, 48.99783]],
    columns=[ID_STATION, "longitude", "latitude"])
echangeurs = pd.DataFrame(
    [['E000001', 2.5305161526, 49.0136341017],
     ['E000003', 2.5511088602, 49.0864274846]], 
    columns=['id_pool', 'longitude', 'latitude'])
expected3000 = pd.DataFrame(
    [['FRHPCPNF059704TOTEMSA', 'E000001'],
    ['FRTSLP436970', 'E000001'],
    ['FRELCP12953864', 'E000001'],
    ['FRHPCPNF059705TOTEM', 'E000001'],
    ['FRGYMP735569', 'E000001'],
    ['FRELCP12953963', 'E000001'],
    ['FRVIAP122064', 'E000001'],
    ['FRGYMP735571', 'E000003'],
    ['FRFASP11568718', 'E000003'],
    ['FRIOYP13530862', 'E000003'],
    ['FRPD1PKFCWTZ', 'E000003'],
    ['FRFASP11568719', 'E000003'],
    ['FRECHPCRXMB', 'E000003'],
    ['FRVIAP152005', 'E000003'],
    ['FRELCP13050707', 'E000003'],
    ['FRMILP11437562', 'E000003']], 
    columns= [ID_STATION, ID_POOL])
expected500 = pd.DataFrame(
    [['FRGYMP735571', 'E000003'],
    ['FRIOYP13530862', 'E000003'],
    ['FRPD1PKFCWTZ', 'E000003'],
    ['FRMILP11437562', 'E000003']], 
    columns= [ID_STATION, ID_POOL])

RAYON_ECHANGEUR = 100000000
assert stations_echangeurs(stations, echangeurs, RAYON_ECHANGEUR)[[ID_STATION, ID_POOL]].sort_values(by=ID_STATION).reset_index(drop=True).equals(expected3000.sort_values(by=ID_STATION).reset_index(drop=True))
RAYON_ECHANGEUR = 3000
assert stations_echangeurs(stations, echangeurs, RAYON_ECHANGEUR)[[ID_STATION, ID_POOL]].sort_values(by=ID_STATION).reset_index(drop=True).equals(expected3000.sort_values(by=ID_STATION).reset_index(drop=True))
RAYON_ECHANGEUR = 500
assert stations_echangeurs(stations, echangeurs, RAYON_ECHANGEUR)[[ID_STATION, ID_POOL]].sort_values(by=ID_STATION).reset_index(drop=True).equals(expected500.sort_values(by=ID_STATION).reset_index(drop=True))
RAYON_ECHANGEUR = 3000

In [4]:
def surfaces_echangeurs(
    points_stations: gpd.GeoDataFrame,
    echangeurs: gpd.GeoDataFrame,
) -> gpd.GeoDataFrame:
    """Return the convex hull surfaces of stations and echangeurs for each pool."""

    points = pd.concat([points_stations[[ID_POOL, "geometry"]], echangeurs[[ID_POOL, "geometry"]]], ignore_index=True)

    surfaces = points.to_crs(epsg=2154).dissolve(by=ID_POOL)
    surfaces["geometry"] = surfaces.geometry.convex_hull.buffer(BUFFER_ECHANGEUR)
  
    return surfaces.to_crs(epsg=4326).reset_index()

In [5]:
def init_pools(file_path: Path) -> gpd.GeoDataFrame:
    """Initialize pools from a geojson file."""
    return gpd.read_file(file_path).rename(columns={"id_aire": "id_pool"})

In [6]:
dmr_file = "../saturation/data_DMR/aires_dmr_2026-08-03.geojson"
pools = init_pools(dmr_file)
echangeurs = pools[pools[ID_POOL].str[:1] == NaturePool.ECHANGEUR.value]
len(echangeurs)

1086

In [ ]:
def read_statics(day: date, min_power: float) -> pd.DataFrame:
    """Read static data for pocs and stations."""
    date_statics = f"{day.day:02d}-{day.month:02d}-{day.year}"
    e5_str = pd.read_csv(f"../saturation/data_DMR_e2_e3/e5_{date_statics}.csv")["extras"][0]
    statics = pd.DataFrame(json.loads(e5_str))
    statics["unite"] = statics["id_pdc_itinerance"].str[:5]
    return statics #[statics["puissance_nominale"] >= min_power]


In [8]:
min_power = 0 # 50
statics = read_statics(day, min_power)
#statics = statics[statics["puissance_nominale"] >= min_power]
len(statics)


01-08-2026
e5  4310979


67596

In [9]:
rayon_echangeur = 3000
ech_station = stations_echangeurs(statics, echangeurs, rayon_echangeur)
len(ech_station), len(statics)

(3026, 67596)

In [10]:
ech_station[ech_station["id_pool"]=="E000001"]

,id_station_itinerance,geometry,id_pool,distance
24,FRHPCPNF059704TOTEMSA,POINT (2.52299 49.00004),E000001,1608.466571
331,FRTSLP436970,POINT (2.51569 48.99112),E000001,2728.914910
391,FRELCP12953864,POINT (2.51549 48.98999),E000001,2850.004795
1281,FRVIAP122064,POINT (2.51408 48.99783),E000001,2129.799147
1324,FRHPCPNF059705TOTEM,POINT (2.53734 49.00931),E000001,692.886145
1653,FRELCP12953963,POINT (2.51781 48.99893),E000001,1881.284533
1929,FRGYMP735569,POINT (2.56084 49.00859),E000001,2288.653224
2438,FRSIGPSIGE491,POINT (2.51714 49.00224),E000001,1601.085863


In [11]:
surf = surfaces_echangeurs(ech_station, echangeurs)
surf

,id_pool,geometry
0,E000001,"POLYGON ((2.51621 48.98923, 2.51609 48.98918, ..."
1,E000002,"POLYGON ((2.54184 49.03385, 2.54184 49.03377, ..."
2,E000003,"POLYGON ((2.55434 49.07163, 2.55432 49.07155, ..."
3,E000004,"POLYGON ((2.56367 49.13352, 2.56367 49.13344, ..."
4,E000005,"POLYGON ((2.6122 49.20618, 2.61228 49.2061, 2...."
...,...,...
1081,E001082,"POLYGON ((6.76636 45.91557, 6.76635 45.91548, ..."
1082,E001083,"POLYGON ((6.76292 45.92582, 6.76291 45.92573, ..."
1083,E001084,"POLYGON ((6.74838 45.92827, 6.7485 45.92827, 6..."
1084,E001085,"POLYGON ((1.79542 42.57962, 1.79541 42.57953, ..."


In [12]:
refnat = {'location': [46.3, 2.3], 'zoom_start': 7, 'tiles': TILES_IGN, 'attr': ATTR_IGN}
#refnat  = {'tiles':'CartoDB positron', 'location': [46.3, 2.3], 'zoom_start': 7}
carte = surf.explore(m=folium.Map(**refnat), color='green', name='aires', show=True)
##carte.save("aires.html")
#webbrowser.open(Path("aires.html").resolve().as_uri())
carte